# Oscar — rough workings

My own working: things tried, dead ends, and code that may or may not make the
report. The unit reads this as evidence of contribution, so keep everything and
date it. It does not need to run top to bottom.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"

## 26 Sep 2026 — loading the Taiwan credit-card data

The file is already in `data/raw/` (notebook `report/01` downloads it). It is an
old-style Excel `.xls`, so pandas needs the `xlrd` package to read it. Row 0 of
the sheet is a title, so `header=1` tells pandas the column names are on the
second row. `index_col="ID"` uses the customer ID as the row label instead of
adding a 0, 1, 2... index.

In [13]:
taiwan = pd.read_excel(RAW / "taiwan-credit-default" / "default of credit card clients.xls",
                       header=1, index_col="ID")

print(taiwan.shape)   # (rows, columns) - expect (30000, 24)

print(taiwan.columns)
taiwan.head()

(30000, 24)
Index(['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2',
       'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default payment next month'],
      dtype='str')


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
ID,,,,,,,,,,,,,,,,,,,,,
1,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
2,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
3,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
4,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
5,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


Note that there is no PAY_1 for some reason.

## 26 Sep 2026 — rough EDA on the Taiwan data

First look: is the file clean, how common is default, do the columns hold the
values the UCI description says they should, and which columns look related to
default? Uses `taiwan` from the cell above.

In [4]:
# The label's name is long, so give it a short alias.
Y = "default payment next month"

# Are any values missing? Are any rows exact copies of another row?
print("missing values:", taiwan.isna().sum().sum())
print("duplicate rows:", taiwan.duplicated().sum())
taiwan.dtypes.value_counts()   # what types pandas gave the columns

missing values: 0
duplicate rows: 35


int64    24
Name: count, dtype: int64

`duplicated()` compares whole rows *ignoring the ID* (the ID is the index, not
a column), so a duplicate here means two customers with identical values in all
24 columns. Worth asking: same person entered twice, or genuinely identical?

In [5]:
# How common is default? normalize=True gives proportions instead of counts.
taiwan[Y].value_counts(normalize=True)

default payment next month
0    0.7788
1    0.2212
Name: proportion, dtype: float64

### Do the codes match the documentation?

In [15]:
for col in ["SEX", "EDUCATION", "MARRIAGE", "PAY_0"]:
    print(col, taiwan[col].value_counts().sort_index().to_dict())

SEX {1: 11888, 2: 18112}
EDUCATION {0: 14, 1: 10585, 2: 14030, 3: 4917, 4: 123, 5: 280, 6: 51}
MARRIAGE {0: 54, 1: 13659, 2: 15964, 3: 323}
PAY_0 {-2: 2759, -1: 5686, 0: 14737, 1: 3688, 2: 2667, 3: 322, 4: 76, 5: 26, 6: 11, 7: 9, 8: 19}


There are non +ve values in PAY_* columns. Need to be careful with this as they do not mean a negative delay.

### Discrepancies with documentation

28 Sep 2026. The documentation is the UCI page (Yeh, 2016) and the paper that
introduced the data (Yeh & Lien, 2009); both describe the variables in the same
words. Page numbers are from the paper. Every action happens in code after
loading, so `data/raw` stays unchanged.

| # | Discrepancy | Evidence | Source | Action |
|---|---|---|---|---|
| 1 | Column names aren't the documented ones, and there's no `PAY_1` | Docs use `X1`-`X23`, `Y`. The file's first header row has `X6` above `PAY_0` and `X7`...`X11` above `PAY_2`...`PAY_6`, so no month is missing; the short names skip 1. `PAY_0`, `BILL_AMT1`, `PAY_AMT1` are all September 2005 | Yeh & Lien (2009, p. 2475) | **Rename `PAY_0` to `PAY_1`** after loading: `taiwan.rename(columns={"PAY_0": "PAY_1"})`. Say so in the report, as readers may expect `PAY_0` |
| 2 | `PAY_*` has undocumented values -2 and 0 | Docs: "-1 = pay duly; 1 = payment delay for one month; ...; 9 = ... nine months and above". In `PAY_0`, -2 appears 2,759 times and 0 appears 14,737 times (58%). `PAY_0` never reaches 9 | Yeh & Lien (2009, p. 2475); Yeh (2016) | **Keep them as categories; don't drop.** Dropping loses a lot of data and biases the sample (default about 13% for these codes vs about 70% for 2+ months late). One-hot encode, or split into "months late" + category. To do: count rows with -2/0 in *any* month; agree with Alex and Louis |
| 3 | `EDUCATION` has undocumented values 0, 5, 6 | Docs: 1 = graduate school, 2 = university, 3 = high school, 4 = others. File also has 0 (14), 5 (280), 6 (51) | Yeh & Lien (2009, p. 2475) | **Merge 0, 5, 6 into 4 ("others")**: 345 people, about 1%. Usual choice with this dataset; give the reason in the report |
| 4 | `MARRIAGE` has undocumented value 0 | Docs: 1 = married, 2 = single, 3 = others. File also has 0 (54) | Yeh & Lien (2009, p. 2475) | **Merge 0 into 3 ("others")**: 54 people. Same reasoning as 3 |
| 5 | Sample size doesn't match the paper | Paper: "25,000 observations, 5529 observations (22.12%) ... with default payment". File: 30,000 rows, same 22.12% | Yeh & Lien (2009, p. 2475) | **Leave it.** Use all 30,000 rows. Note as a limitation: we can't exactly reproduce the paper's setup |
| 6 | The paper contradicts itself on the default rate | Paper says "non-risky (87.88%)", but 100% - 22.12% = 77.88%, which matches the file. Probably a typo | Yeh & Lien (2009, p. 2475) | **Leave it.** Mention only if we quote the paper's percentages |

Also worth knowing (not a discrepancy): the target, `default payment next
month`, refers to October 2005. The paper says the data is "payment data in
October, 2005" (Yeh & Lien, 2009, p. 2475), the month after `PAY_0`.

**References**

- Yeh, I-C. and Lien, C-h. (2009). The comparisons of data mining techniques for
  the predictive accuracy of probability of default of credit card clients.
  *Expert Systems with Applications*, 36(2), 2473-2480.
  https://doi.org/10.1016/j.eswa.2007.12.020
- Yeh, I-C. (2016). *Default of Credit Card Clients* [dataset]. UCI Machine
  Learning Repository. https://doi.org/10.24432/C55S3H